In [33]:
# Import neccesary libraries.
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import randint

# Current Approach in improving performance
 - Remove Outliers
 - Remove high dependent variables
 - Use regularization methods
 - Use deep neural networks
 - check assumption
 - Transform the response variable for stable distribution
 - Create more measures (utilities) from the data (e.g. Star Powers)

In [3]:
df = pd.read_csv("data/movie_data_encoded.csv")

In [ ]:
y = df['revenue']
X = df.drop(['Unnamed: 0', 'name', 'genres', 'directors', 'writers','actors', 'original_language', 'production_companies',
       'release_date','keywords', 'revenue'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [34]:
dummy_model = np.full_like(y_test, y.mean()) 
RMSE = mean_squared_error(y_test, dummy_model)
R2_dummy = r2_score(y_test, dummy_model)
print(f'The RMSE value for the Dummy Model is {RMSE}, and {R2_dummy}')

The RMSE value for the Dummy Model is 1704861315192150.5, and -0.00044957113756738387


In [30]:
# Check VIF of each variable
vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i)
              for i in range(len(X.columns))]
print(vif)

c:\Users\wue77\anaconda3\envs\movie_env\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
c:\Users\wue77\anaconda3\envs\movie_env\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


               feature           VIF
0        averageRating  1.549631e+00
1             numVotes  1.710622e+00
2               budget  2.075551e+00
3              runtime  1.475852e+00
4        trailer_views  2.041825e+00
5        trailer_likes  2.480008e+00
6      titleType_movie           inf
7    titleType_tvMovie           inf
8       director_score  1.834575e+00
9         writer_score  1.505465e+00
10         actor_score  1.258697e+00
11      is_documentary  1.328555e+00
12       genres_Action  2.531592e+02
13        genres_Adult  1.131196e+00
14    genres_Adventure  1.180175e+02
15    genres_Animation  1.539711e+01
16    genres_Biography  7.496518e+01
17       genres_Comedy  2.828009e+02
18        genres_Crime  9.364854e+01
19  genres_Documentary  2.313249e+01
20        genres_Drama  2.345431e+02
21       genres_Family  3.517328e+00
22      genres_Fantasy  7.674835e+00
23    genres_Film-Noir  1.128926e+00
24      genres_History  1.636455e+00
25       genres_Horror  6.010365e+01
2

In [ ]:
RFR_tune = RandomForestRegressor()
param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(1,20),
    'min_samples_leaf': randint(1, 10)
}

rand_search = RandomizedSearchCV(RFR_tune, param_distributions=param_dist, n_iter=10, cv=5, random_state=42)

# Fit the random search object to the data.
rand_search.fit(X_train, y_train.values.ravel())
rand_search.best_estimator_

RandomForestRegressor(max_depth=19, n_estimators=264)

In [31]:
RFR = RandomForestRegressor(max_depth=19, n_estimators=264)
RFR.fit(X_train_scaled, y_train.values.ravel())
RMSE = mean_squared_error(y_test, RFR.predict(X_test_scaled))
R2_RFR = r2_score(y_test, RFR.predict(X_test_scaled))
print(f'The RMSE value for the Random Forest Regression Model is {RMSE}, and {R2_RFR}')

The RMSE value for the Random Forest Regression Model is 789455630179112.0, and 0.5367303254483119
